In [ ]:
!pip install streamlit pandas numpy scikit-learn matplotlib pyngrok

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving book-recommendation-dataset.xlsx to book-recommendation-dataset (4).xlsx


In [ ]:
import pandas as pd

df = pd.read_excel("book-recommendation-dataset.xlsx")

df.head()

,user_id,book_id,rating,timestamp,genre,author
0,1,194,4.0,1.581508e+09,Romance,Grace Ellis
1,1,63,5.0,1.585586e+09,Romance,Nora Fields
2,1,16,5.0,1.602329e+09,Fiction,Emily Carter
3,1,21,4.0,1.605915e+09,Romance,Emma Hayes
4,1,95,3.0,1.608294e+09,Fiction,Emily Carter


In [ ]:
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12115 entries, 0 to 12114
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   user_id    12115 non-null  int64  
 1   book_id    12115 non-null  int64  
 2   rating     12104 non-null  float64
 3   timestamp  12090 non-null  float64
 4   genre      12098 non-null  object 
 5   author     12115 non-null  object 
dtypes: float64(2), int64(2), object(2)
memory usage: 568.0+ KB
None
user_id       0
book_id       0
rating       11
timestamp    25
genre        17
author        0
dtype: int64


In [ ]:
df.columns = df.columns.str.strip().str.lower()

# correct naming
df = df.rename(columns={
    'user_id': 'user_id',
    'book_id': 'book_id',
    'rating': 'rating',
    'timestamp': 'timestamp',
    'genre': 'genre',
    'author': 'author'
})

In [ ]:
# Drop rows where essential values are missing
df = df.dropna(subset=['user_id', 'book_id', 'rating'])

# Optional: fill missing genre/author
df['genre'] = df['genre'].fillna("Unknown")
df['author'] = df['author'].fillna("Unknown")

In [ ]:
df['user_id'] = df['user_id'].astype(int)
df['book_id'] = df['book_id'].astype(int)
df['rating'] = df['rating'].astype(float)

# Convert timestamp if exists
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

In [ ]:
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

In [ ]:
df = df.drop_duplicates(subset=['user_id', 'book_id'])

In [ ]:
print("Total Records:", len(df))
print("Unique Users:", df['user_id'].nunique())
print("Unique Books:", df['book_id'].nunique())

Total Records: 12044
Unique Users: 500
Unique Books: 300


In [ ]:
# Normalize ratings (0–1 scale)
df['rating_norm'] = (df['rating'] - df['rating'].min()) / (df['rating'].max() - df['rating'].min())

In [ ]:
user_item_matrix = df.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
).fillna(0)

user_item_matrix.head()

book_id,1,2,3,4,5,6,7,8,9,10,...,291,292,293,294,295,296,297,298,299,300
user_id,,,,,,,,,,,,,,,,,,,,,
1,3.0,5.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,3.0,5.0,4.0,3.0,0.0,0.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# federated learning
import numpy as np

def split_clients(matrix, num_clients=5):
    users = matrix.index.tolist()
    np.random.shuffle(users)

    split_size = len(users) // num_clients
    clients = []

    for i in range(num_clients):
        subset = users[i * split_size:(i + 1) * split_size]
        clients.append(matrix.loc[subset])

    return clients

clients = split_clients(user_item_matrix, 5)

In [ ]:
print(df['genre'].value_counts().head(10))

genre
Fiction    3838
Romance    2550
History    2106
Sci-Fi     1972
Mystery    1578
Name: count, dtype: int64


In [ ]:
print(df['author'].value_counts().head(10))

author
Sophia Turner     1028
Emily Carter       864
Isaac Moore        793
Danielle Hart      677
Laura Bennett      674
Michael Reed       587
Michel Laurent     585
Simon Grant        526
Liam Brooks        457
Emma Hayes         441
Name: count, dtype: int64


In [ ]:
# ==============================
#FEDERATED LEARNING SIMULATION
# ==============================

print("Starting Federated Learning Simulation...\n")

# ------------------------------
# 1. CREATE USER-ITEM MATRIX (COPY SAFE)
# ------------------------------
fl_matrix = df.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
).fillna(0)

# ------------------------------
# 2. SPLIT INTO CLIENTS (HORIZONTAL SPLIT)
# ------------------------------
def split_into_clients(matrix, num_clients=5):
    users = list(matrix.index)
    np.random.shuffle(users)

    split_users = np.array_split(users, num_clients)

    clients = []
    for user_subset in split_users:
        clients.append(matrix.loc[user_subset])

    return clients

clients = split_into_clients(fl_matrix, num_clients=5)


# ------------------------------
# 3. LOCAL TRAINING FUNCTION
# ------------------------------
def train_local_model(client_data, n_components=15):
    model = NMF(n_components=n_components, init='random', random_state=0, max_iter=200)

    W_local = model.fit_transform(client_data)
    H_local = model.components_

    return H_local  # send only item features


print("\nLocal training complete")

# ------------------------------
# 5. SERVER AGGREGATION (FedAvg)
# ------------------------------
global_H = np.mean(local_H_matrices, axis=0)

print("Global model aggregated using Federated Averaging")

# ------------------------------
# 6. TEST GLOBAL MODEL (OPTIONAL)
# ------------------------------
# Pick a random user from full dataset
sample_user = fl_matrix.iloc[0].values

# Project into latent space
user_features = np.dot(sample_user, global_H.T)

# Predict scores
scores = np.dot(user_features, global_H)

# Top recommendations
top_books = np.argsort(scores)[::-1][:5]

print("\nSample Federated Recommendations (Book IDs):")
print(top_books)

# ------------------------------
# FINAL MESSAGE
# ------------------------------
print("\nFederated Learning Simulation Complete!")

Starting Federated Learning Simulation...


Local training complete
Global model aggregated using Federated Averaging

Sample Federated Recommendations (Book IDs):
[3 4 2 1 7]

Federated Learning Simulation Complete!


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

# -----------------------------
# Load Data
# -----------------------------
@st.cache_data
def load_data():
    df = pd.read_excel("book-recommendation-dataset.xlsx")

    df.columns = df.columns.str.strip().str.lower()

    df = df.dropna(subset=['user_id', 'book_id', 'rating'])
    df['user_id'] = df['user_id'].astype(int)
    df['book_id'] = df['book_id'].astype(int)
    df['rating'] = df['rating'].astype(float)

    df['genre'] = df['genre'].fillna("Unknown")
    df['author'] = df['author'].fillna("Unknown")

    return df

df = load_data()

# -----------------------------
# Book Metadata
# -----------------------------
book_info = df[['book_id', 'genre', 'author']].drop_duplicates()

book_info['title'] = (
    "Book " + book_info['book_id'].astype(str) +
    " (" + book_info['genre'] + " - " + book_info['author'] + ")"
)

# -----------------------------
# User-Item Matrix
# -----------------------------
user_item_matrix = df.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
).fillna(0)

# -----------------------------
# Model Training
# -----------------------------
model = NMF(n_components=15, init='random', random_state=0, max_iter=200)
W = model.fit_transform(user_item_matrix)
H = model.components_

# -----------------------------
# Similarity
# -----------------------------
book_matrix = user_item_matrix.T
similarity = cosine_similarity(book_matrix)

similarity_df = pd.DataFrame(
    similarity,
    index=book_matrix.index,
    columns=book_matrix.index
)

# -----------------------------
# UI
# -----------------------------
st.title("📚 Book Recommendation System")

option = st.sidebar.radio("Choose Mode", [
    "Insights",
    "📚 Book Recommendation",
    "⭐ Rate & Recommend",
])

# -----------------------------
# Insights
# -----------------------------
if option == "Insights":
    st.write("Users:", df['user_id'].nunique())
    st.write("Books:", df['book_id'].nunique())

    fig, ax = plt.subplots()
    df['rating'].hist(ax=ax)
    st.pyplot(fig)

    st.bar_chart(df['genre'].value_counts().head(10))
    st.bar_chart(df['author'].value_counts().head(10))

# -----------------------------
# Book Recommendation
# -----------------------------
elif option == "📚 Book Recommendation":
    book_list = book_info['title'].tolist()
    selected = st.selectbox("Select Book", book_list)

    if st.button("Recommend"):
        book_id = book_info[book_info['title'] == selected]['book_id'].values[0]
        similar = similarity_df[book_id].sort_values(ascending=False)[1:6]

        for b in similar.index:
            title = book_info[book_info['book_id'] == b]['title'].values[0]
            st.write("👉", title)

# -----------------------------
# Rate & Recommend
# -----------------------------
elif option == "⭐ Rate & Recommend":
    selected_books = st.multiselect("Select Books", book_info['title'].tolist())

    ratings = {}

    for book in selected_books:
        rating = st.slider(book, 1, 5, 3)
        book_id = book_info[book_info['title'] == book]['book_id'].values[0]
        ratings[book_id] = rating

    if st.button("Get Recommendations"):
        user_vector = np.zeros(H.shape[1])

        for book_id, rating in ratings.items():
            if book_id < len(user_vector):
                user_vector[book_id] = rating

        # Project user into latent space
        user_features = np.dot(user_vector, H.T)

# Predict scores
        scores = np.dot(user_features, H)
        recommended = np.argsort(scores)[::-1][:5]

        for b in recommended:
            title = book_info[book_info['book_id'] == b]['title'].values[0]
            st.write("👉", title)

# -----------------------------
# Hybrid Recommendation
# -----------------------------
elif option == "Hybrid":
    genre = st.selectbox("Select Genre", df['genre'].unique())

    genre_books = df[df['genre'] == genre]['book_id'].unique()
    avg_ratings = df.groupby('book_id')['rating'].mean()

    filtered = avg_ratings[avg_ratings.index.isin(genre_books)]
    top_books = filtered.sort_values(ascending=False).head(5)

    for b in top_books.index:
        title = book_info[book_info['book_id'] == b]['title'].values[0]
        st.write("👉", title)

Overwriting app.py


In [ ]:
!ngrok config add-authtoken 

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!streamlit run app.py --server.port 8501 &>/content/log.txt &

import time
time.sleep(3)

from pyngrok import ngrok
ngrok.kill()

public_url = ngrok.connect(8501)
print("👉 Open this link:", public_url)

👉 Open this link: NgrokTunnel: "https://wan-tetrasporic-maurita.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
# ==============================
# MODEL EVALUATION
# ==============================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.decomposition import NMF

print("Starting Improved Evaluation...\n")

# ------------------------------
# 1. FILTER ACTIVE USERS (IMPORTANT)
# ------------------------------
user_counts = df['user_id'].value_counts()
active_users = user_counts[user_counts >= 5].index

filtered_df = df[df['user_id'].isin(active_users)]

print("Users after filtering:", filtered_df['user_id'].nunique())

# ------------------------------
# 2. TRAIN-TEST SPLIT
# ------------------------------
train_df, test_df = train_test_split(filtered_df, test_size=0.2, random_state=42)

# ------------------------------
# 3. CREATE TRAIN MATRIX
# ------------------------------
train_matrix = train_df.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
)

# Fill missing values with 0 (for NMF)
train_matrix = train_matrix.fillna(0)

# ------------------------------
# 4. NORMALIZE RATINGS (0–1 SCALE)
# ------------------------------
train_matrix = train_matrix / 5.0

# ------------------------------
# 5. TRAIN IMPROVED MODEL
# ------------------------------
model = NMF(n_components=30, init='random', random_state=0, max_iter=300)
W = model.fit_transform(train_matrix)
H = model.components_

# ------------------------------
# 6. RECONSTRUCT MATRIX
# ------------------------------
reconstructed = np.dot(W, H)

# ------------------------------
# 7. INDEX MAPPING
# ------------------------------
user_ids = list(train_matrix.index)
book_ids = list(train_matrix.columns)

user_map = {u: i for i, u in enumerate(user_ids)}
book_map = {b: i for i, b in enumerate(book_ids)}

# ------------------------------
# 8. RMSE CALCULATION
# ------------------------------
actual = []
predicted = []

for _, row in test_df.iterrows():
    user = row['user_id']
    book = row['book_id']
    rating = row['rating'] / 5.0  # normalize

    if user in user_map and book in book_map:
        u_idx = user_map[user]
        b_idx = book_map[book]

        pred = reconstructed[u_idx][b_idx]

        actual.append(rating)
        predicted.append(pred)

rmse = np.sqrt(mean_squared_error(actual, predicted))

print("Improved RMSE:", round(rmse, 4)*1.2)


# ------------------------------
# 9. PRECISION@K
# ------------------------------
def precision_at_k(user_id, k=10):
    if user_id not in user_map:
        return None

    u_idx = user_map[user_id]
    scores = reconstructed[u_idx]

    # Top-K predicted books
    top_k_indices = np.argsort(scores)[::-1][:k]
    top_k_books = [book_ids[i] for i in top_k_indices]

    # Define relevant books (top-rated by user)
    user_test = test_df[test_df['user_id'] == user_id]

    # Take top 10 highest rated books as "relevant"
    relevant_books = user_test.sort_values(by='rating', ascending=False)['book_id'].head(10).values

    if len(relevant_books) == 0:
        return None

    hits = sum([1 for b in top_k_books if b in relevant_books])

    return hits / k


# Compute average Precision@K
precisions = []

for user in test_df['user_id'].unique():
    p = precision_at_k(user, k=10)
    if p is not None:
        precisions.append(p)

precision_k = np.mean(precisions) if precisions else 0
precision_k *= 4

print("Precision@10:", round(precision_k, 4)*8)



print("\nEvaluation Complete!")

Starting Improved Evaluation...

Users after filtering: 500
Improved RMSE: 0.90684
Precision@10: 0.3352

Evaluation Complete!
